# 因子暴露和因子收益率完整教程：从定义到两种方法的对比

## 📚 教学目标
1. 理解**因子暴露 (Factor Exposure / Factor Loading / β)** 的两种含义
2. 区分**排序法因子收益率**与**回归法因子收益率**的定义和计算
3. 在微型数据集上**手算**两种因子收益率，建立直觉
4. 理解为什么两种方法得到的因子收益率**不是同一个东西**
5. 通过大样本模拟，量化两种方法的关系与差异

**参考书**：《因子投资：方法与实践》（石川）第 2.3 节
**教学策略**：先用极小数据集让你"看见"每一步计算，再扩展到真实规模

---

## 1. 场景设定：因子暴露到底是什么？

### 🎯 一个直觉问题

在第 2.1 节中，我们学会了**排序法**：按因子值分组，构建多空组合，计算 Spread。
在第 2.2 节中，我们学会了**回归法**：用截面回归估计因子的风险溢价 $\lambda$。

但你有没有想过：

> **排序法和回归法得到的"因子收益率"是同一个东西吗？**

答案是：**不是！** 它们在概念和数值上都不同。

要理解这个区别，我们必须先搞清楚一个核心概念：**因子暴露 (Factor Exposure)**。

### 📖 书中的定义

> 因子暴露（factor exposure），也称因子载荷（factor loading），衡量的是资产对某个因子的敏感程度。
> 它既可以通过时间序列回归的回归系数 $\beta$ 来度量，也可以直接用股票的某个特征值（如市值的对数）来衡量。

### 🔗 本节的逻辑链条

```
因子暴露 β 的定义
    ↓
两种理解: 回归系数 β vs 股票特征值
    ↓
排序法: 按 β 排序分组 → Long-Short Spread = f_sort
    ↓
回归法: 截面回归 r_i = λ₀ + λ₁·β_i + e_i → λ₁ = f_reg
    ↓
对比: f_sort ≠ f_reg → 为什么？什么时候近似？
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

# 设置中文字体和样式
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 因子暴露的两种理解

### 📐 理解一：时间序列回归系数 $\beta$

给定一个因子收益率时间序列 $f_t$，对每只股票 $i$ 做时间序列回归：

$$r_{i,t} = \alpha_i + \beta_i \cdot f_t + \varepsilon_{i,t}$$

其中：
- $r_{i,t}$ = 股票 $i$ 在时期 $t$ 的收益率
- $f_t$ = 因子收益率（如市场组合收益率）
- $\beta_i$ = 股票 $i$ 的因子暴露（回归系数）
- $\varepsilon_{i,t}$ = 残差

$\beta_i$ 的含义：**因子收益率每变动 1%，股票 $i$ 的收益率平均变动 $\beta_i$%**。

### 📐 理解二：股票特征值（标准化后）

在实际研究中，我们经常直接用股票的**某个特征值**来衡量因子暴露：

$$\beta_i = \text{Standardize}(\text{Characteristic}_i) = \frac{X_i - \bar{X}}{\sigma_X}$$

例如：
- 市值因子暴露 ≈ 标准化的 $\log(\text{MarketCap})$
- 价值因子暴露 ≈ 标准化的 $\text{Book-to-Market}$
- 动量因子暴露 ≈ 标准化的过去 12 个月累计收益率

### 💡 两种理解的联系

| 比较 | 时间序列回归 $\beta$ | 特征值标准化 |
|------|---------------------|-------------|
| 数据需求 | 需要因子收益率时间序列 | 只需要截面特征数据 |
| 估计精度 | 受限于样本长度 | 无需估计，直接观测 |
| 适用场景 | CAPM、Fama-French 因子 | 公司特征因子（Size, B/M 等）|
| 本质 | 对因子的敏感程度 | 因子的代理变量 |

---

## 3. 微型数据集：10 只股票的因子暴露

### 🎯 场景

假设市场只有 **10 只股票**。我们已知每只股票的**因子暴露 $\beta_i$**（可以理解为标准化后的某个特征值），以及它们**当期的收益率 $r_i$**。

### 📐 数据生成过程 (DGP)

$$r_i = \alpha + \beta_i \cdot f + \varepsilon_i$$

其中：
- $\beta_i$ 为预先设定的因子暴露
- $f = 0.8\%$ 为真实因子收益率（未知，待估计）
- $\varepsilon_i \sim N(0, 1.5)$ 为个股特异性噪声
- $\alpha = 1.0\%$ 为截距

In [ ]:
# ========== 构造 10 只股票的微型数据集 ==========
np.random.seed(42)

# 预设因子暴露：从 -1.5 到 1.5，代表标准化后的特征值
betas = np.array([-1.5, -1.1, -0.7, -0.3, -0.1, 0.1, 0.4, 0.8, 1.2, 1.6])

# DGP 参数
alpha_true = 1.0   # 截距
f_true = 0.8       # 真实因子收益率
noise = np.random.normal(0, 1.5, 10)  # 个股噪声

# 生成收益率
returns = alpha_true + betas * f_true + noise

# 构建 DataFrame
df_micro = pd.DataFrame({
    'Stock': [f'S{i+1:02d}' for i in range(10)],
    'Beta': betas,
    'Return': np.round(returns, 2)
})

print("📋 10 只股票的微型数据集：")
print("─" * 40)
print(df_micro.to_string(index=False))
print(f"\n📐 DGP 参数（真实值，实际研究中未知）：")
print(f"  α = {alpha_true}%, f = {f_true}%")
print(f"  真实模型: r_i = {alpha_true} + {f_true} × β_i + ε_i")
print(f"\n💡 β 越大的股票，预期收益越高（因为 f > 0）")
print(f"   但噪声的存在让这个关系不完美")

---

## 4. 排序法因子收益率：$f_{\text{sort}}$

### 📖 回顾 2.1 节的方法

排序法的步骤：
1. 按因子暴露 $\beta_i$ 对股票排序
2. 分成 $L$ 组（如 5 组）
3. 做多高暴露组，做空低暴露组
4. **Spread = $R_{\text{Long}} - R_{\text{Short}}$**

这个 Spread 就是**排序法因子收益率** $f_{\text{sort}}$。

### 📐 数学定义

$$f_{\text{sort}} = \frac{1}{n_{\text{High}}} \sum_{i \in \text{High}} r_i - \frac{1}{n_{\text{Low}}} \sum_{i \in \text{Low}} r_i$$

### 💡 关键特征

- 这是一个**Dollar-Neutral**（资金中性）的多空组合收益
- 它依赖于**分组方式**（分几组、断点怎么选）
- 它**不利用中间组**的信息（只用两端的极端组）

In [ ]:
# ========== 排序法因子收益率 ==========
# 10 只股票分成 5 组，每组 2 只
N_GROUPS = 5
df_sorted = df_micro.sort_values('Beta').reset_index(drop=True)

# 分组
labels = [f'Q{i}' for i in range(1, N_GROUPS + 1)]
df_sorted['Group'] = pd.qcut(df_sorted['Beta'], q=N_GROUPS, labels=labels)

print("📊 步骤 1: 按因子暴露 β 排序并分组")
print("─" * 55)
for q in labels:
    group = df_sorted[df_sorted['Group'] == q]
    stocks = ', '.join(group['Stock'].values)
    betas_str = ', '.join([f'{b:.1f}' for b in group['Beta'].values])
    avg_ret = group['Return'].mean()
    print(f"  {q}: [{stocks}]  β = [{betas_str}]  R̄ = {avg_ret:.2f}%")

# 计算各组平均收益
group_rets = df_sorted.groupby('Group')['Return'].mean()

# 排序法因子收益率 = 做多高β组 - 做空低β组
f_sort = group_rets['Q5'] - group_rets['Q1']

print(f"\n📊 步骤 2: 构建多空组合")
print(f"  做多 Q5 (高β): R_long  = {group_rets['Q5']:.2f}%")
print(f"  做空 Q1 (低β): R_short = {group_rets['Q1']:.2f}%")
print(f"")
print(f"  📐 f_sort = R_long − R_short")
print(f"           = {group_rets['Q5']:.2f}% − {group_rets['Q1']:.2f}%")
print(f"           = {f_sort:.2f}%")
print(f"")
print(f"  💡 排序法因子收益率 f_sort = {f_sort:.2f}%")
print(f"     这是一个 Dollar-Neutral 多空组合的收益")
print(f"     真实因子收益率 f_true = {f_true}%")

---

## 5. 回归法因子收益率：$\hat{\lambda}_1$ 

### 📖 回顾 2.2 节的方法

回归法的核心是**截面回归 (Cross-Sectional Regression)**：

$$r_i = \lambda_0 + \lambda_1 \cdot \beta_i + e_i$$

其中：
- $r_i$ = 股票 $i$ 的当期收益率
- $\beta_i$ = 股票 $i$ 的因子暴露
- $\lambda_0$ = 截距（零暴露资产的预期收益）
- $\lambda_1$ = **因子风险溢价 (Factor Risk Premium)**
- $e_i$ = 残差

### 💡 关键特征

- $\lambda_1$ 衡量的是：**因子暴露每增加 1 个单位，预期收益率增加多少**
- 它利用了**所有股票**的信息（不只是两端的极端组）
- 它是一个**纯回归系数**，不是组合收益率
- 当 $\beta_i$ 就是标准化特征值时，这就是 **Fama-MacBeth 回归**的第二步

In [ ]:
# ========== 回归法因子收益率 ==========
print("📊 步骤 1: 截面回归 r_i = λ₀ + λ₁ × β_i + e_i")
print("─" * 55)

# 准备回归数据
X = sm.add_constant(df_micro['Beta'])
y = df_micro['Return']

# OLS 回归
model = sm.OLS(y, X).fit()

lambda_0 = model.params['const']
lambda_1 = model.params['Beta']
r_squared = model.rsquared

print(f"  回归结果：")
print(f"  r_i = {lambda_0:.4f} + {lambda_1:.4f} × β_i")
print(f"")
print(f"  λ₀ (截距)     = {lambda_0:.4f}%")
print(f"  λ₁ (风险溢价) = {lambda_1:.4f}%")
print(f"  R² = {r_squared:.4f}")
print(f"")
print(f"📊 步骤 2: 解读")
print(f"  📐 回归法因子收益率 λ₁ = {lambda_1:.4f}%")
print(f"  含义: 因子暴露每增加 1 个单位，预期收益率增加 {lambda_1:.4f}%")
print(f"")
print(f"  💡 真实因子收益率 f_true = {f_true}%")
print(f"     回归估计 λ₁ = {lambda_1:.4f}% (受噪声影响，不完全等于 f_true)")

In [ ]:
# ========== 可视化：截面回归拟合 ==========
fig, ax = plt.subplots(figsize=(10, 6))

# 散点图
colors_scatter = plt.cm.RdYlGn(np.linspace(0.15, 0.85, 10))
for i, (_, row) in enumerate(df_micro.iterrows()):
    ax.scatter(row['Beta'], row['Return'], c=[colors_scatter[i]], 
               s=120, edgecolors='black', zorder=5)
    ax.annotate(row['Stock'], (row['Beta'], row['Return']),
                textcoords='offset points', xytext=(6, 6), fontsize=9)

# 回归线
beta_line = np.linspace(df_micro['Beta'].min() - 0.2, df_micro['Beta'].max() + 0.2, 100)
ret_line = lambda_0 + lambda_1 * beta_line
ax.plot(beta_line, ret_line, 'r-', linewidth=2.5, 
        label=f'OLS: r = {lambda_0:.2f} + {lambda_1:.2f} x $\\beta$ (R\u00b2={r_squared:.3f})')

# 真实关系线
ret_true_line = alpha_true + f_true * beta_line
ax.plot(beta_line, ret_true_line, 'b--', linewidth=2, alpha=0.6,
        label=f'True: r = {alpha_true:.1f} + {f_true:.1f} x $\\beta$')

# 标注关键信息
textstr = f'$\\lambda_1$ (OLS) = {lambda_1:.4f}%\nf_true = {f_true:.4f}%\n$R^2$ = {r_squared:.3f}'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=11,
        verticalalignment='top', bbox=props)

ax.set_xlabel('Factor Exposure ($\\beta$)', fontsize=12)
ax.set_ylabel('Return (%)', fontsize=12)
ax.set_title('Cross-Sectional Regression: Return vs Factor Exposure', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  红色实线 = OLS 拟合的截面回归线，斜率 λ₁ = {lambda_1:.4f}%")
print(f"  蓝色虚线 = 真实关系线，斜率 f_true = {f_true}%")
print(f"  两条线不完全重合，因为只有 10 个数据点 + 噪声")

---

## 6. 对比：排序法 vs 回归法因子收益率

### 🎯 核心问题

现在我们有了同一组数据上的两个估计：
- 排序法因子收益率 $f_{\text{sort}}$
- 回归法因子收益率 $\hat{\lambda}_1$

它们是同一个东西吗？让我们对比看看。

In [ ]:
# ========== 直接对比两种因子收益率 ==========
print("📊 两种因子收益率的直接对比（10 只股票，单期）")
print("═" * 55)
print(f"")
print(f"  排序法因子收益率  f_sort = {f_sort:.4f}%")
print(f"  回归法因子收益率  λ₁     = {lambda_1:.4f}%")
print(f"  真实因子收益率    f_true = {f_true:.4f}%")
print(f"")
print(f"  差异: f_sort − λ₁ = {f_sort - lambda_1:.4f}%")
print(f"")
print(f"💡 关键发现: f_sort ≠ λ₁ ！")
print(f"")
print(f"📐 为什么不同？")
print(f"")
print(f"  1. 排序法只用了 Q1 和 Q5 的信息（4 只股票）")
print(f"     回归法用了所有 10 只股票的信息")
print(f"")
print(f"  2. 排序法是等权多空组合的收益（Dollar-Neutral）")
print(f"     回归法是 β 每增加 1 单位对应的收益增量（Risk Premium）")
print(f"")
print(f"  3. 排序法的结果取决于分组数量和断点选择")
print(f"     回归法的结果取决于 β 的分布形态")
print(f"")
print(f"═" * 55)

In [ ]:
# ========== 可视化对比 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左图：排序法视角 ---
ax1 = axes[0]
group_ret_vals = [group_rets[q] for q in labels]
colors_bar = plt.cm.RdYlGn(np.linspace(0.15, 0.85, N_GROUPS))
bars = ax1.bar(labels, group_ret_vals, color=colors_bar, edgecolor='black', alpha=0.85)

for bar, v in zip(bars, group_ret_vals):
    va = 'bottom' if v >= 0 else 'top'
    offset = 0.1 if v >= 0 else -0.1
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + offset,
             f'{v:.2f}%', ha='center', va=va, fontweight='bold')

# 标注 Spread
ax1.annotate('', xy=(4.3, group_rets['Q5']), xytext=(4.3, group_rets['Q1']),
             arrowprops=dict(arrowstyle='<->', color='red', linewidth=2))
ax1.text(4.5, (group_rets['Q5'] + group_rets['Q1'])/2, 
         f'Spread\n= {f_sort:.2f}%', color='red', fontweight='bold', fontsize=11)

ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.set_xlabel('Portfolio Group', fontsize=12)
ax1.set_ylabel('Average Return (%)', fontsize=12)
ax1.set_title(f'Sort Method: f_sort = {f_sort:.2f}%', fontsize=13, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# --- 右图：回归法视角 ---
ax2 = axes[1]
ax2.scatter(df_micro['Beta'], df_micro['Return'], c='steelblue', 
            s=100, edgecolors='black', zorder=5)
beta_line = np.linspace(-2, 2, 100)
ax2.plot(beta_line, lambda_0 + lambda_1 * beta_line, 'r-', linewidth=2.5,
         label=f'$\\lambda_1$ = {lambda_1:.2f}%')

# 标注斜率
ax2.annotate('', xy=(1.0, lambda_0 + lambda_1 * 1.0), 
             xytext=(0.0, lambda_0 + lambda_1 * 0.0),
             arrowprops=dict(arrowstyle='->', color='red', linewidth=2))
ax2.text(0.5, lambda_0 + lambda_1 * 0.5 + 0.5, 
         f'Slope = $\\lambda_1$ = {lambda_1:.2f}%',
         color='red', fontweight='bold', fontsize=11)

ax2.set_xlabel('Factor Exposure ($\\beta$)', fontsize=12)
ax2.set_ylabel('Return (%)', fontsize=12)
ax2.set_title(f'Regression Method: $\\lambda_1$ = {lambda_1:.2f}%', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：排序法只看两端 — Q5 做多, Q1 做空 → Spread = {f_sort:.2f}%")
print(f"  右图：回归法拟合所有点 — 斜率 λ₁ = {lambda_1:.2f}%")
print(f"  两种方法对\"因子收益率\"的定义本质不同！")

---

## 7. 扩展到 50 只股票：看差异更清楚

10 只股票太少，噪声太大。现在我们扩展到 **50 只股票**，让两种方法的差异更清晰。

In [ ]:
# ========== 50 只股票的单期对比 ==========
np.random.seed(42)
N = 50
N_GROUPS_50 = 5

# 生成因子暴露：正态分布
betas_50 = np.random.normal(0, 1, N)

# DGP
f_true_50 = 0.8
alpha_50 = 1.0
noise_50 = np.random.normal(0, 3, N)
returns_50 = alpha_50 + betas_50 * f_true_50 + noise_50

df_50 = pd.DataFrame({'Beta': betas_50, 'Return': returns_50})

# --- 排序法 ---
labels_50 = [f'Q{i}' for i in range(1, N_GROUPS_50 + 1)]
df_50['Group'] = pd.qcut(df_50['Beta'], q=N_GROUPS_50, labels=labels_50)
group_rets_50 = df_50.groupby('Group')['Return'].mean()
f_sort_50 = group_rets_50['Q5'] - group_rets_50['Q1']

# --- 回归法 ---
X_50 = sm.add_constant(df_50['Beta'])
model_50 = sm.OLS(df_50['Return'], X_50).fit()
lambda_1_50 = model_50.params['Beta']
lambda_0_50 = model_50.params['const']

# --- 平均 β 差异 ---
avg_beta_Q5 = df_50[df_50['Group'] == 'Q5']['Beta'].mean()
avg_beta_Q1 = df_50[df_50['Group'] == 'Q1']['Beta'].mean()
delta_beta = avg_beta_Q5 - avg_beta_Q1

print(f"📊 50 只股票的两种因子收益率")
print(f"═" * 55)
print(f"  排序法 f_sort = {f_sort_50:.4f}%")
print(f"  回归法 λ₁     = {lambda_1_50:.4f}%")
print(f"  真实值 f_true = {f_true_50:.4f}%")
print(f"")
print(f"📐 为什么不同？看 β 的分布：")
print(f"  Q1 组的平均 β = {avg_beta_Q1:.4f}")
print(f"  Q5 组的平均 β = {avg_beta_Q5:.4f}")
print(f"  Δβ = β_Q5 − β_Q1 = {delta_beta:.4f}")
print(f"")
print(f"💡 关键公式：")
print(f"  f_sort ≈ λ₁ × Δβ")
print(f"  验证: {lambda_1_50:.4f} × {delta_beta:.4f} = {lambda_1_50 * delta_beta:.4f}")
print(f"  实际: f_sort = {f_sort_50:.4f}")
print(f"")
print(f"  📐 排序法 Spread = 回归斜率 × 高低组的 β 差异")
print(f"     只有当 Δβ = 1 时，f_sort 才等于 λ₁")

In [ ]:
# ========== 可视化：50 只股票的排序法 vs 回归法 ==========
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- 左图：所有股票 + 回归线 + 组均值 ---
ax1 = axes[0]
scatter = ax1.scatter(df_50['Beta'], df_50['Return'], c='lightgray', 
                       s=40, edgecolors='gray', alpha=0.6, label='Individual Stocks')

# 各组均值
group_avg_beta = df_50.groupby('Group')['Beta'].mean()
group_avg_ret = df_50.groupby('Group')['Return'].mean()
colors_q = plt.cm.RdYlGn(np.linspace(0.15, 0.85, N_GROUPS_50))
for i, q in enumerate(labels_50):
    ax1.scatter(group_avg_beta[q], group_avg_ret[q], c=[colors_q[i]], 
                s=200, edgecolors='black', zorder=10, marker='D')
    ax1.annotate(q, (group_avg_beta[q], group_avg_ret[q]),
                 textcoords='offset points', xytext=(8, 8), fontsize=10, fontweight='bold')

# 回归线
beta_range = np.linspace(df_50['Beta'].min() - 0.3, df_50['Beta'].max() + 0.3, 100)
ax1.plot(beta_range, lambda_0_50 + lambda_1_50 * beta_range, 'r-', linewidth=2.5,
         label=f'OLS: slope $\\lambda_1$ = {lambda_1_50:.3f}%')

# 连接 Q1 和 Q5
ax1.plot([group_avg_beta['Q1'], group_avg_beta['Q5']], 
         [group_avg_ret['Q1'], group_avg_ret['Q5']], 
         'b--', linewidth=2, label=f'Sort: Q1-Q5 slope = {f_sort_50/delta_beta:.3f}%')

ax1.set_xlabel('Factor Exposure ($\\beta$)', fontsize=12)
ax1.set_ylabel('Return (%)', fontsize=12)
ax1.set_title('50 Stocks: Sort vs Regression', fontsize=14, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# --- 右图：两种方法的结果对比 ---
ax2 = axes[1]
method_names = ['f_sort\n(Sort Method)', '$\\lambda_1$\n(Regression)', 'f_true\n(True Value)']
method_values = [f_sort_50, lambda_1_50, f_true_50]
bar_colors = ['#3498db', '#e74c3c', '#2ecc71']
bars = ax2.bar(method_names, method_values, color=bar_colors, edgecolor='black', alpha=0.85, width=0.5)

for bar, v in zip(bars, method_values):
    va = 'bottom' if v >= 0 else 'top'
    offset = 0.03 if v >= 0 else -0.03
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + offset,
             f'{v:.4f}%', ha='center', va=va, fontweight='bold', fontsize=11)

ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.set_ylabel('Factor Return Estimate (%)', fontsize=12)
ax2.set_title('Comparison of Factor Return Estimates', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：灰色圆点 = 个股，菱形 = 各组均值，红色线 = OLS 回归线")
print(f"        蓝色虚线连接 Q1 和 Q5 → 它的斜率 ≠ OLS 的斜率")
print(f"  右图：排序法的 f_sort 和回归法的 λ₁ 在数值上不同")
print(f"        排序法的数值更大，因为 Δβ > 1（两端极端组的 β 差异大于 1）")

---

## 8. 数学关系：$f_{\text{sort}}$ 与 $\lambda_1$ 的联系

### 📐 精确关系

排序法因子收益率可以表示为：

$$f_{\text{sort}} = \bar{r}_{\text{High}} - \bar{r}_{\text{Low}}$$

如果截面回归的模型是正确的（$r_i = \lambda_0 + \lambda_1 \beta_i + e_i$），那么：

$$f_{\text{sort}} \approx \lambda_1 \times (\bar{\beta}_{\text{High}} - \bar{\beta}_{\text{Low}}) = \lambda_1 \times \Delta\bar{\beta}$$

其中 $\Delta\bar{\beta}$ 是高暴露组与低暴露组的**平均因子暴露之差**。

### 💡 关键推论

1. **当 $\Delta\bar{\beta} = 1$ 时**：$f_{\text{sort}} = \lambda_1$（两种方法等价）
2. **当 $\Delta\bar{\beta} > 1$ 时**：$f_{\text{sort}} > \lambda_1$（排序法高估，因为极端组 β 差异大）
3. **当 $\Delta\bar{\beta} < 1$ 时**：$f_{\text{sort}} < \lambda_1$（排序法低估）

### 📖 书中要点

> 排序法得到的因子收益率本质上是**多空组合的美元收益**，它等于回归法风险溢价乘以高低组的因子暴露差。
> 两种方法不可直接比较大小，因为它们度量的不是同一个概念。

In [ ]:
# ========== 验证数学关系 ==========
print("📊 验证: f_sort ≈ λ₁ × Δβ")
print("═" * 55)

# 不同分组数量下的对比
print(f"  不同分组方式下的对比：")
print(f"  {'分组数':>6} {'f_sort':>10} {'λ₁':>10} {'Δβ':>8} {'λ₁×Δβ':>10} {'误差':>10}")
print(f"  {'─'*6} {'─'*10} {'─'*10} {'─'*8} {'─'*10} {'─'*10}")

for n_g in [2, 3, 5, 10]:
    if n_g > N // 2:
        continue
    g_labels = [f'Q{i}' for i in range(1, n_g + 1)]
    df_50_temp = df_50.copy()
    df_50_temp['Group_temp'] = pd.qcut(df_50_temp['Beta'], q=n_g, labels=g_labels)
    
    g_rets = df_50_temp.groupby('Group_temp')['Return'].mean()
    g_betas = df_50_temp.groupby('Group_temp')['Beta'].mean()
    
    fs = g_rets.iloc[-1] - g_rets.iloc[0]  # f_sort
    db = g_betas.iloc[-1] - g_betas.iloc[0]  # Δβ
    predicted = lambda_1_50 * db
    error = fs - predicted
    
    print(f"  {n_g:>6} {fs:>10.4f} {lambda_1_50:>10.4f} {db:>8.4f} {predicted:>10.4f} {error:>10.4f}")

print(f"\n💡 观察：")
print(f"  1. λ₁ 不随分组数变化（回归用所有数据）")
print(f"  2. f_sort 随分组数变化（分组越少 → Δβ 越大 → f_sort 越大）")
print(f"  3. λ₁ × Δβ 很好地近似了 f_sort")
print(f"  4. 误差来自中间组残差的不完全抵消")

---

## 9. 大样本模拟：200 只股票 × 60 个月

### 🎯 目标

现在我们扩展到**真实规模**，每个月：
1. 生成 200 只股票的截面数据
2. 同时计算排序法和回归法的因子收益率
3. 收集 60 个月的时间序列
4. 对比两种方法的时间序列行为

### 📐 DGP

$$r_{i,t} = \alpha + \beta_i \cdot f_t + \varepsilon_{i,t}$$

其中：
- $\beta_i \sim N(0, 1)$：因子暴露（标准化特征值）
- $f_t \sim N(0.5, 2)$：因子收益率（均值为正，波动较大）
- $\varepsilon_{i,t} \sim N(0, 3)$：个股噪声
- $\alpha = 1.0$：截距

In [ ]:
# ========== 大样本模拟 ==========
np.random.seed(42)
N_STOCKS = 200
N_MONTHS = 60
N_GROUPS_SIM = 5

# 存储结果
f_sort_series = []      # 排序法因子收益率
lambda_1_series = []    # 回归法因子收益率
f_true_series = []      # 真实因子收益率
delta_beta_series = []  # 高低组 β 差异

print(f"📊 模拟参数：")
print(f"  股票数量: {N_STOCKS} 只/月")
print(f"  时间跨度: {N_MONTHS} 个月")
print(f"  分组数量: {N_GROUPS_SIM} 组")
print(f"  DGP: r_i = 1.0 + β_i × f_t + ε_i")
print(f"       β_i ~ N(0, 1), f_t ~ N(0.5, 2), ε_i ~ N(0, 3)")
print(f"")
print(f"🔄 开始逐月计算...")
print(f"")

for t in range(N_MONTHS):
    # --- 生成截面数据 ---
    betas_t = np.random.normal(0, 1, N_STOCKS)      # 因子暴露
    f_t = np.random.normal(0.5, 2)                    # 当月因子收益率
    eps_t = np.random.normal(0, 3, N_STOCKS)          # 个股噪声
    returns_t = 1.0 + betas_t * f_t + eps_t           # 收益率
    
    month_df = pd.DataFrame({'beta': betas_t, 'return': returns_t})
    
    # --- 排序法 ---
    g_labels = [f'Q{i}' for i in range(1, N_GROUPS_SIM + 1)]
    month_df['group'] = pd.qcut(month_df['beta'], q=N_GROUPS_SIM, labels=g_labels)
    g_rets = month_df.groupby('group')['return'].mean()
    g_betas_avg = month_df.groupby('group')['beta'].mean()
    
    fs_t = g_rets['Q5'] - g_rets['Q1']  # 排序法因子收益率
    db_t = g_betas_avg['Q5'] - g_betas_avg['Q1']  # Δβ
    
    # --- 回归法 ---
    X_t = sm.add_constant(month_df['beta'])
    model_t = sm.OLS(month_df['return'], X_t).fit()
    lam1_t = model_t.params['beta']  # 回归法因子收益率
    
    # 存储
    f_sort_series.append(fs_t)
    lambda_1_series.append(lam1_t)
    f_true_series.append(f_t)
    delta_beta_series.append(db_t)
    
    # 打印前 3 个月
    if t < 3:
        print(f"  月份 {t+1}:")
        print(f"    真实 f_t       = {f_t:.4f}%")
        print(f"    排序法 f_sort  = {fs_t:.4f}%")
        print(f"    回归法 λ₁      = {lam1_t:.4f}%")
        print(f"    Δβ (Q5-Q1)    = {db_t:.4f}")
        print(f"    λ₁ × Δβ       = {lam1_t * db_t:.4f} (≈ f_sort)")
        print()

f_sort_arr = np.array(f_sort_series)
lambda_1_arr = np.array(lambda_1_series)
f_true_arr = np.array(f_true_series)
delta_beta_arr = np.array(delta_beta_series)

print(f"  ... (省略第 4~{N_MONTHS} 月) ...")
print(f"\n✅ 完成！共收集 {N_MONTHS} 个月的数据")

---

## 10. 三种因子收益率的时间序列对比

现在我们有三条时间序列：
- $f_{\text{true},t}$：真实因子收益率（DGP 中设定的，实际研究中不可观测）
- $f_{\text{sort},t}$：排序法因子收益率
- $\hat{\lambda}_{1,t}$：回归法因子收益率

它们之间有什么关系？

In [ ]:
# ========== 时间序列统计对比 ==========
print("📊 三种因子收益率的统计对比")
print("═" * 65)
print(f"  {'指标':>12} {'f_true':>12} {'f_sort':>12} {'λ₁':>12}")
print(f"  {'─'*12} {'─'*12} {'─'*12} {'─'*12}")
print(f"  {'均值':>12} {f_true_arr.mean():>12.4f} {f_sort_arr.mean():>12.4f} {lambda_1_arr.mean():>12.4f}")
print(f"  {'标准差':>12} {f_true_arr.std():>12.4f} {f_sort_arr.std():>12.4f} {lambda_1_arr.std():>12.4f}")
print(f"  {'最小值':>12} {f_true_arr.min():>12.4f} {f_sort_arr.min():>12.4f} {lambda_1_arr.min():>12.4f}")
print(f"  {'最大值':>12} {f_true_arr.max():>12.4f} {f_sort_arr.max():>12.4f} {lambda_1_arr.max():>12.4f}")
print(f"")
print(f"📐 相关系数矩阵：")
corr_df = pd.DataFrame({
    'f_true': f_true_arr,
    'f_sort': f_sort_arr,
    'lambda_1': lambda_1_arr
})
print(corr_df.corr().round(4).to_string())
print(f"")

# Δβ 的统计
print(f"📐 高低组 β 差异 Δβ 的统计：")
print(f"  均值: {delta_beta_arr.mean():.4f}")
print(f"  标准差: {delta_beta_arr.std():.4f}")
print(f"")
print(f"💡 关键发现：")
print(f"  1. f_sort 的均值 ({f_sort_arr.mean():.4f}) ≈ λ₁ 的均值 ({lambda_1_arr.mean():.4f}) × Δβ 的均值 ({delta_beta_arr.mean():.4f})")
print(f"     验证: {lambda_1_arr.mean():.4f} × {delta_beta_arr.mean():.4f} = {lambda_1_arr.mean() * delta_beta_arr.mean():.4f}")
print(f"  2. f_sort 的标准差 ({f_sort_arr.std():.4f}) > λ₁ 的标准差 ({lambda_1_arr.std():.4f})")
print(f"     因为 f_sort = λ₁ × Δβ，被 Δβ 放大了")
print(f"  3. 三者高度正相关，但数值不同")

In [ ]:
# ========== 可视化：时间序列对比 ==========
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

months = np.arange(1, N_MONTHS + 1)

# --- 图1: 三条时间序列 ---
ax1 = axes[0, 0]
ax1.plot(months, f_true_arr, 'b-', linewidth=1.5, alpha=0.8, label='$f_{true}$')
ax1.plot(months, lambda_1_arr, 'r-', linewidth=1.5, alpha=0.8, label='$\\lambda_1$ (Regression)')
ax1.plot(months, f_sort_arr, 'g--', linewidth=1.5, alpha=0.7, label='$f_{sort}$ (Sort)')
ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.set_xlabel('Month', fontsize=11)
ax1.set_ylabel('Factor Return (%)', fontsize=11)
ax1.set_title('Three Factor Return Estimates Over Time', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# --- 图2: f_sort vs λ₁ 散点图 ---
ax2 = axes[0, 1]
ax2.scatter(lambda_1_arr, f_sort_arr, c='steelblue', s=40, edgecolors='black', alpha=0.7)
# 拟合线
z = np.polyfit(lambda_1_arr, f_sort_arr, 1)
x_line = np.linspace(lambda_1_arr.min(), lambda_1_arr.max(), 100)
ax2.plot(x_line, np.poly1d(z)(x_line), 'r-', linewidth=2, 
         label=f'OLS: slope = {z[0]:.3f}')
# 45度线
lim = max(abs(lambda_1_arr).max(), abs(f_sort_arr).max()) + 1
ax2.plot([-lim, lim], [-lim, lim], 'k--', alpha=0.3, label='45-degree line')

corr_sl = np.corrcoef(lambda_1_arr, f_sort_arr)[0, 1]
textstr = f'Correlation = {corr_sl:.4f}\nSlope = {z[0]:.4f}\nMean $\\Delta\\beta$ = {delta_beta_arr.mean():.4f}'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax2.text(0.02, 0.98, textstr, transform=ax2.transAxes, fontsize=11,
         verticalalignment='top', bbox=props)

ax2.set_xlabel('$\\lambda_1$ (Regression Factor Return)', fontsize=11)
ax2.set_ylabel('$f_{sort}$ (Sort Factor Return)', fontsize=11)
ax2.set_title('Sort vs Regression Factor Returns', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10, loc='lower right')
ax2.grid(alpha=0.3)

# --- 图3: λ₁ vs f_true 散点图 ---
ax3 = axes[1, 0]
ax3.scatter(f_true_arr, lambda_1_arr, c='#e74c3c', s=40, edgecolors='black', alpha=0.7)
z2 = np.polyfit(f_true_arr, lambda_1_arr, 1)
x_line2 = np.linspace(f_true_arr.min(), f_true_arr.max(), 100)
ax3.plot(x_line2, np.poly1d(z2)(x_line2), 'r-', linewidth=2,
         label=f'OLS: slope = {z2[0]:.3f}')
ax3.plot([-lim, lim], [-lim, lim], 'k--', alpha=0.3, label='45-degree line')

corr_tl = np.corrcoef(f_true_arr, lambda_1_arr)[0, 1]
textstr2 = f'Correlation = {corr_tl:.4f}'
ax3.text(0.02, 0.98, textstr2, transform=ax3.transAxes, fontsize=11,
         verticalalignment='top', bbox=props)

ax3.set_xlabel('$f_{true}$ (True Factor Return)', fontsize=11)
ax3.set_ylabel('$\\lambda_1$ (Regression Estimate)', fontsize=11)
ax3.set_title('Regression vs True Factor Return', fontsize=13, fontweight='bold')
ax3.legend(fontsize=10, loc='lower right')
ax3.grid(alpha=0.3)

# --- 图4: f_sort vs f_true 散点图 ---
ax4 = axes[1, 1]
ax4.scatter(f_true_arr, f_sort_arr, c='#2ecc71', s=40, edgecolors='black', alpha=0.7)
z3 = np.polyfit(f_true_arr, f_sort_arr, 1)
x_line3 = np.linspace(f_true_arr.min(), f_true_arr.max(), 100)
ax4.plot(x_line3, np.poly1d(z3)(x_line3), 'r-', linewidth=2,
         label=f'OLS: slope = {z3[0]:.3f}')
ax4.plot([-lim, lim], [-lim, lim], 'k--', alpha=0.3, label='45-degree line')

corr_ts = np.corrcoef(f_true_arr, f_sort_arr)[0, 1]
textstr3 = f'Correlation = {corr_ts:.4f}'
ax4.text(0.02, 0.98, textstr3, transform=ax4.transAxes, fontsize=11,
         verticalalignment='top', bbox=props)

ax4.set_xlabel('$f_{true}$ (True Factor Return)', fontsize=11)
ax4.set_ylabel('$f_{sort}$ (Sort Estimate)', fontsize=11)
ax4.set_title('Sort vs True Factor Return', fontsize=13, fontweight='bold')
ax4.legend(fontsize=10, loc='lower right')
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  图1：三条因子收益率时间序列同步波动，但数值不同")
print(f"  图2：f_sort 与 λ₁ 高度正相关 (ρ = {corr_sl:.4f})，但散点偏离 45 度线")
print(f"       拟合斜率 = {z[0]:.4f} ≈ 平均 Δβ = {delta_beta_arr.mean():.4f}")
print(f"  图3：λ₁ 是 f_true 的无偏估计 (斜率 ≈ 1，相关 = {corr_tl:.4f})")
print(f"  图4：f_sort 与 f_true 的关系受 Δβ 放大 (斜率 = {z3[0]:.4f} > 1)")

---

## 11. T 检验：两种方法的统计推断对比

### 🎯 问题

两种方法都给出了因子收益率的时间序列，我们分别做 T 检验：

- **排序法**: $H_0: E[f_{\text{sort}}] = 0$ → $t_{\text{sort}} = \bar{f}_{\text{sort}} / SE(\bar{f}_{\text{sort}})$
- **回归法**: $H_0: E[\lambda_1] = 0$ → $t_{\text{reg}} = \bar{\lambda}_1 / SE(\bar{\lambda}_1)$

两个 T 检验的**结论是否一致**？T 值是否相同？

In [ ]:
# ========== T 检验对比 ==========

# 排序法 T 检验
sort_mean = f_sort_arr.mean()
sort_std = f_sort_arr.std(ddof=1)
sort_se = sort_std / np.sqrt(N_MONTHS)
sort_t = sort_mean / sort_se
sort_p = 2 * (1 - stats.t.cdf(abs(sort_t), df=N_MONTHS-1))

# 回归法 T 检验 (Fama-MacBeth)
reg_mean = lambda_1_arr.mean()
reg_std = lambda_1_arr.std(ddof=1)
reg_se = reg_std / np.sqrt(N_MONTHS)
reg_t = reg_mean / reg_se
reg_p = 2 * (1 - stats.t.cdf(abs(reg_t), df=N_MONTHS-1))

# scipy 验证
t_sort_scipy, p_sort_scipy = stats.ttest_1samp(f_sort_arr, 0)
t_reg_scipy, p_reg_scipy = stats.ttest_1samp(lambda_1_arr, 0)

print("📊 T 检验对比：排序法 vs 回归法")
print("═" * 60)
print(f"  {'指标':>14} {'排序法 f_sort':>16} {'回归法 λ₁':>16}")
print(f"  {'─'*14} {'─'*16} {'─'*16}")
print(f"  {'均值':>14} {sort_mean:>16.4f} {reg_mean:>16.4f}")
print(f"  {'标准差':>14} {sort_std:>16.4f} {reg_std:>16.4f}")
print(f"  {'标准误':>14} {sort_se:>16.4f} {reg_se:>16.4f}")
print(f"  {'T统计量':>14} {sort_t:>16.4f} {reg_t:>16.4f}")
print(f"  {'P值':>14} {sort_p:>16.6f} {reg_p:>16.6f}")
print(f"")

print(f"🔬 scipy 验证：")
print(f"  排序法: t = {t_sort_scipy:.4f}, P = {p_sort_scipy:.6f} ✅")
print(f"  回归法: t = {t_reg_scipy:.4f}, P = {p_reg_scipy:.6f} ✅")
print(f"")

print(f"💡 关键发现：")
print(f"  1. 两种方法的 T 值不同: t_sort = {sort_t:.2f}, t_reg = {reg_t:.2f}")
print(f"  2. 但两种方法的结论一致: {'两者都显著' if sort_p < 0.05 and reg_p < 0.05 else '结论不一致'}")
print(f"  3. T 值的比值 ≈ {sort_t / reg_t:.4f}")
print(f"     这反映了 f_sort 与 λ₁ 的信噪比差异")

In [ ]:
# ========== 可视化 T 检验对比 ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- 图1: 两种因子收益率分布对比 ---
ax1 = axes[0]
ax1.hist(f_sort_arr, bins=15, alpha=0.6, color='#3498db', edgecolor='black',
         density=True, label=f'$f_{{sort}}$ (mean={sort_mean:.3f})')
ax1.hist(lambda_1_arr, bins=15, alpha=0.6, color='#e74c3c', edgecolor='black',
         density=True, label=f'$\\lambda_1$ (mean={reg_mean:.3f})')
ax1.axvline(x=0, color='black', linewidth=1.5, linestyle='--', label='$H_0$: mean=0')
ax1.axvline(x=sort_mean, color='#3498db', linewidth=2, linestyle='-')
ax1.axvline(x=reg_mean, color='#e74c3c', linewidth=2, linestyle='-')
ax1.set_xlabel('Factor Return (%)', fontsize=11)
ax1.set_ylabel('Density', fontsize=11)
ax1.set_title('Distribution: Sort vs Regression', fontsize=13, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# --- 图2: 累积收益 ---
ax2 = axes[1]
cum_sort = np.cumsum(f_sort_arr)
cum_reg = np.cumsum(lambda_1_arr)
cum_true = np.cumsum(f_true_arr)
ax2.plot(months, cum_true, 'b-', linewidth=2, alpha=0.5, label='$f_{true}$ (Cumulative)')
ax2.plot(months, cum_reg, 'r-', linewidth=2, label='$\\lambda_1$ (Cumulative)')
ax2.plot(months, cum_sort, 'g--', linewidth=2, label='$f_{sort}$ (Cumulative)')
ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.set_xlabel('Month', fontsize=11)
ax2.set_ylabel('Cumulative Return (%)', fontsize=11)
ax2.set_title('Cumulative Factor Returns', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

# --- 图3: T 值对比柱状图 ---
ax3 = axes[2]
t_vals = [sort_t, reg_t]
t_labels = ['$t_{sort}$', '$t_{reg}$']
t_colors = ['#3498db', '#e74c3c']
bars = ax3.bar(t_labels, t_vals, color=t_colors, edgecolor='black', alpha=0.85, width=0.4)

for bar, v in zip(bars, t_vals):
    ax3.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
             f'{v:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=13)

ax3.axhline(y=2.0, color='orange', linewidth=2, linestyle='--', label='Critical Value (|t|=2)')
ax3.axhline(y=0, color='black', linewidth=0.8)
ax3.set_ylabel('T-Statistic', fontsize=12)
ax3.set_title('T-Statistics Comparison', fontsize=13, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  图1：蓝色 = f_sort 分布（更宽），红色 = λ₁ 分布（更窄）")
print(f"       f_sort 的均值更大，但标准差也更大")
print(f"  图2：累积收益走势相似但幅度不同，f_sort > λ₁ > f_true 的累积值")
print(f"  图3：两种方法的 T 值可以不同，因为信噪比不同")

---

## 12. 深入理解：什么情况下两种方法差异更大？

### 💡 决定差异的关键因素

$$f_{\text{sort}} = \lambda_1 \times \Delta\bar{\beta}$$

所以差异的大小取决于 $\Delta\bar{\beta}$ 与 1 的偏离程度。

**$\Delta\bar{\beta}$ 与 1 偏离更大的情况**：
1. β 的分布越分散（$\sigma_\beta$ 越大）→ $\Delta\bar{\beta}$ 越大
2. 分组数量越少 → 极端组越极端 → $\Delta\bar{\beta}$ 越大
3. β 的分布越偏斜 → $\Delta\bar{\beta}$ 越不对称

In [ ]:
# ========== 实验：β 的分散程度如何影响两种方法的差异 ==========
np.random.seed(42)
N_STOCKS_EXP = 200
N_MONTHS_EXP = 60
N_GROUPS_EXP = 5
sigma_betas = [0.3, 0.5, 1.0, 1.5, 2.0, 3.0]  # β 的标准差

results_exp = []

for sig_b in sigma_betas:
    fs_list = []
    lam_list = []
    db_list = []
    
    for t in range(N_MONTHS_EXP):
        betas_exp = np.random.normal(0, sig_b, N_STOCKS_EXP)
        f_exp = np.random.normal(0.5, 2)
        eps_exp = np.random.normal(0, 3, N_STOCKS_EXP)
        ret_exp = 1.0 + betas_exp * f_exp + eps_exp
        
        df_exp = pd.DataFrame({'beta': betas_exp, 'return': ret_exp})
        df_exp['group'] = pd.qcut(df_exp['beta'], q=N_GROUPS_EXP,
                                   labels=[f'Q{i}' for i in range(1, N_GROUPS_EXP+1)])
        g_r = df_exp.groupby('group')['return'].mean()
        g_b = df_exp.groupby('group')['beta'].mean()
        
        fs_list.append(g_r.iloc[-1] - g_r.iloc[0])
        db_list.append(g_b.iloc[-1] - g_b.iloc[0])
        
        X_exp = sm.add_constant(df_exp['beta'])
        lam_list.append(sm.OLS(df_exp['return'], X_exp).fit().params['beta'])
    
    results_exp.append({
        'sigma_beta': sig_b,
        'mean_f_sort': np.mean(fs_list),
        'mean_lambda': np.mean(lam_list),
        'mean_delta_beta': np.mean(db_list),
        'ratio': np.mean(fs_list) / np.mean(lam_list) if np.mean(lam_list) != 0 else np.nan
    })

df_results = pd.DataFrame(results_exp)

print("📊 β 分散程度对两种因子收益率的影响")
print("═" * 70)
print(f"  {'σ_β':>6} {'f_sort均值':>12} {'λ₁均值':>10} {'Δβ均值':>10} {'f_sort/λ₁':>10}")
print(f"  {'─'*6} {'─'*12} {'─'*10} {'─'*10} {'─'*10}")
for _, row in df_results.iterrows():
    print(f"  {row['sigma_beta']:>6.1f} {row['mean_f_sort']:>12.4f} {row['mean_lambda']:>10.4f} {row['mean_delta_beta']:>10.4f} {row['ratio']:>10.4f}")

print(f"\n💡 关键发现：")
print(f"  1. λ₁ 的均值基本不变（≈ 0.5），因为回归只关注斜率")
print(f"  2. f_sort 随 σ_β 增大而增大，因为 Δβ 也增大")
print(f"  3. f_sort / λ₁ ≈ Δβ，验证了数学关系")
print(f"  4. 当 σ_β = 0.3 时两者最接近，σ_β = 3.0 时差异最大")

In [ ]:
# ========== 可视化实验结果 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左图：两种因子收益率随 σ_β 变化 ---
ax1 = axes[0]
ax1.plot(df_results['sigma_beta'], df_results['mean_f_sort'], 'go-', 
         linewidth=2, markersize=8, label='$\\bar{f}_{sort}$')
ax1.plot(df_results['sigma_beta'], df_results['mean_lambda'], 'rs-', 
         linewidth=2, markersize=8, label='$\\bar{\\lambda}_1$')
ax1.axhline(y=0.5, color='blue', linestyle='--', alpha=0.5, label='True f = 0.5')
ax1.set_xlabel('$\\sigma_{\\beta}$ (Dispersion of Factor Exposure)', fontsize=12)
ax1.set_ylabel('Mean Factor Return (%)', fontsize=12)
ax1.set_title('Effect of $\\beta$ Dispersion on Factor Returns', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# --- 右图：f_sort / λ₁ 的比值 vs Δβ ---
ax2 = axes[1]
ax2.plot(df_results['mean_delta_beta'], df_results['ratio'], 'ko-', 
         linewidth=2, markersize=8)
# 理论线 (ratio = Δβ)
db_range = np.linspace(df_results['mean_delta_beta'].min(), df_results['mean_delta_beta'].max(), 100)
ax2.plot(db_range, db_range, 'r--', linewidth=2, alpha=0.6, label='Theory: ratio = $\\Delta\\beta$')

ax2.set_xlabel('Mean $\\Delta\\bar{\\beta}$ (Q5 - Q1)', fontsize=12)
ax2.set_ylabel('$f_{sort}$ / $\\lambda_1$ Ratio', fontsize=12)
ax2.set_title('Ratio vs $\\Delta\\beta$: Theory Matches', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：λ₁ 几乎不随 β 分散度变化（水平红线）")
print(f"        f_sort 随 β 分散度增大而线性增大（绿色上升线）")
print(f"  右图：f_sort/λ₁ 的比值几乎完美落在 y = Δβ 的理论线上")
print(f"        验证了 f_sort = λ₁ × Δβ 的数学关系")

---

## 13. 两种方法的全面对比

### 📝 对比总结表

| 特性 | 排序法因子收益率 $f_{\text{sort}}$ | 回归法因子收益率 $\hat{\lambda}_1$ |
|------|-------------------------------|-----------------------------------|
| **定义** | 多空组合的收益率 | 截面回归的斜率系数 |
| **含义** | Dollar-Neutral 策略的实际收益 | 单位因子暴露的风险溢价 |
| **利用的信息** | 只用两端极端组（如 Q1, Q5） | 利用所有股票 |
| **受分组影响** | ✅ 是（分组数量、断点选择） | ❌ 否 |
| **受 β 分布影响** | ✅ 是（通过 $\Delta\bar{\beta}$） | ❌ 否（直接估计斜率） |
| **可投资性** | ✅ 高（对应实际交易策略） | ⚠️ 低（理论概念，非实际组合） |
| **数学关系** | $f_{\text{sort}} \approx \lambda_1 \times \Delta\bar{\beta}$ | $\lambda_1 = f_{\text{sort}} / \Delta\bar{\beta}$ |
| **代表文献** | Fama & French (1993) | Fama & MacBeth (1973) |

### 📖 书中要点

> 排序法得到的因子收益率是多空组合的实际美元收益，反映了因子溢价在具体投资策略中的实现。
> 回归法得到的因子收益率（风险溢价 $\lambda$）是因子暴露每增加一个单位的预期收益增量。
> 两者在概念和数值上都不同，不可混淆。

### 💡 何时使用哪种方法？

| 研究目的 | 推荐方法 | 原因 |
|---------|---------|------|
| 构建交易策略 | 排序法 | 直接对应可投资的多空组合 |
| 估计因子风险溢价 | 回归法 | 利用所有数据，更精确 |
| 多因子模型检验 | 回归法 | 可同时控制多个因子 |
| 因子有效性初筛 | 排序法 | 简单直觉，可视化效果好 |
| Fama-French 因子构建 | 排序法 | 学术界标准做法 |

In [ ]:
# ========== 最终汇总报告 ==========
print("=" * 65)
print("📋 因子暴露与因子收益率 — 完整汇总报告")
print("=" * 65)

print(f"\n🎯 研究问题:")
print(f"   排序法和回归法得到的因子收益率是否相同？")

print(f"\n📊 数据概况:")
print(f"   股票数量: {N_STOCKS} 只/月")
print(f"   时间跨度: {N_MONTHS} 个月")
print(f"   真实因子收益率: f ~ N(0.5, 2)")

print(f"\n🧮 统计结果:")
print(f"   排序法 f_sort: 均值 = {sort_mean:.4f}%, t = {sort_t:.2f}, P = {sort_p:.4f}")
print(f"   回归法 λ₁:    均值 = {reg_mean:.4f}%, t = {reg_t:.2f}, P = {reg_p:.4f}")
print(f"   f_sort / λ₁ = {sort_mean / reg_mean:.4f} ≈ Δβ = {delta_beta_arr.mean():.4f}")
print(f"   相关系数 ρ(f_sort, λ₁) = {corr_sl:.4f}")

print(f"\n🎯 结论:")
print(f"  1. 两种方法的因子收益率在数值上不同")
print(f"  2. 它们的关系为: f_sort ≈ λ₁ × Δβ")
print(f"  3. 两种方法高度正相关 (ρ = {corr_sl:.4f})")
print(f"  4. 当 Δβ ≠ 1 时，直接比较两者的数值没有意义")
print(f"  5. 两种方法在统计推断的结论上通常一致")

print("\n" + "=" * 65)

---

## 14. 核心概念回顾

### 📌 因子暴露 (Factor Exposure / Factor Loading / β)
- **定义**: 衡量资产对某个因子的敏感程度
- **两种度量**: 时间序列回归系数 $\beta_i$；标准化后的股票特征值
- **含义**: $\beta_i = 1.5$ 表示因子每变动 1%，该股票预期变动 1.5%

### 📌 排序法因子收益率 ($f_{\text{sort}}$)
- **定义**: 按因子暴露排序分组，多空组合的收益率
- **公式**: $f_{\text{sort}} = \bar{r}_{\text{High}} - \bar{r}_{\text{Low}}$
- **含义**: Dollar-Neutral 多空策略的实际收益
- **特点**: 依赖分组方式，只用极端组信息

### 📌 回归法因子收益率 ($\hat{\lambda}_1$)
- **定义**: 截面回归 $r_i = \lambda_0 + \lambda_1 \beta_i + e_i$ 中的斜率
- **公式**: $\hat{\lambda}_1 = \frac{\sum_i (\beta_i - \bar{\beta})(r_i - \bar{r})}{\sum_i (\beta_i - \bar{\beta})^2}$
- **含义**: 单位因子暴露对应的收益增量（风险溢价）
- **特点**: 利用所有股票信息，不受分组方式影响

### 📌 两种因子收益率的关系
- **公式**: $f_{\text{sort}} \approx \lambda_1 \times \Delta\bar{\beta}$
- **其中**: $\Delta\bar{\beta} = \bar{\beta}_{\text{High}} - \bar{\beta}_{\text{Low}}$
- **含义**: 排序法 Spread = 回归斜率 × 高低组的因子暴露差异
- **判断**: 只有当 $\Delta\bar{\beta} = 1$ 时两者数值相等

### 🔗 完整流程

```
确定因子暴露 β → 排序法：排序分组 → Long-Short Spread = f_sort
                → 回归法：截面回归 → 斜率系数 λ₁ = f_reg
                → 关系: f_sort ≈ λ₁ × Δβ
                → T 检验判断显著性
```

### 📝 检验指标汇总

| 概念 | 排序法 | 回归法 |
|------|--------|--------|
| 因子收益率 | $f_{\text{sort}} = R_{\text{Long}} - R_{\text{Short}}$ | $\hat{\lambda}_1 = \text{OLS slope}$ |
| 含义 | 多空组合收益 | 单位暴露的风险溢价 |
| T 检验 | $t = \bar{f}_{\text{sort}} / SE$ | $t = \bar{\lambda}_1 / SE$ |
| 数据利用 | 仅极端组 | 全部股票 |
| 可投资性 | 高 | 低 |

---

## 15. 常见误区

### ❌ 误区 1: 排序法和回归法的因子收益率是同一个东西
**✓ 正确理解**: 排序法得到的是多空组合的 Dollar-Neutral 收益，回归法得到的是单位因子暴露的风险溢价 $\lambda_1$。两者的关系为 $f_{\text{sort}} \approx \lambda_1 \times \Delta\bar{\beta}$，只有在 $\Delta\bar{\beta} = 1$ 时数值才相等。

### ❌ 误区 2: 因子暴露 β 和因子收益率 f 是同一个概念
**✓ 正确理解**: $\beta_i$ 是股票 $i$ 对因子的**敏感程度**（截面属性），$f$ 是因子本身的**收益率**（时间序列变量）。在 $r_i = \alpha + \beta_i \cdot f + \varepsilon_i$ 中，$\beta_i$ 是已知的系数，$f$ 是要估计的未知量。

### ❌ 误区 3: 回归法比排序法"更好"
**✓ 正确理解**: 两种方法回答不同的问题。排序法构建可投资的因子策略（"这个因子能赚多少钱？"），回归法估计因子的定价能力（"单位暴露值多少风险溢价？"）。选择哪种方法取决于研究目的。

### ❌ 误区 4: 两种方法的 T 值应该相同
**✓ 正确理解**: 虽然两种方法通常给出一致的显著性结论（都显著或都不显著），但 T 值的**数值**不同。这是因为 $f_{\text{sort}}$ 和 $\lambda_1$ 有不同的均值和标准差，信噪比 (Signal-to-Noise Ratio) 不同。

### ❌ 误区 5: 排序法的分组数量不影响因子收益率
**✓ 正确理解**: 分组数量直接影响 $\Delta\bar{\beta}$。分组越少（如 2 组），极端组的 $\bar{\beta}$ 差异越小；分组越多（如 10 组），极端组越极端，$\Delta\bar{\beta}$ 越大。因此排序法的因子收益率 $f_{\text{sort}}$ 会随分组数量变化，而回归法的 $\lambda_1$ 不受影响。